In [4]:
"../data/customer_ltv_segments.csv"

'../data/customer_ltv_segments.csv'

In [5]:
import pandas as pd

df_active = pd.read_csv("../data/customer_churn_predictions.csv")

print(df_active.shape)
print(df_active.head())

(7043, 23)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... StreamingTV  \
0  No phone service             DSL             No  ...          No   
1                No             DSL            Yes  ...          No   
2                No             DSL            Yes  ...          No   
3  No phone service             DSL            Yes  ...          No   
4                No     Fiber optic             No  ...          No   

  StreamingMovies        Contract PaperlessBilling              PaymentMethod  \


In [6]:
print(df_active.columns.tolist())

['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn', 'churn_probability', 'churn_risk']


In [8]:
print(df_active.columns.tolist())

['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn', 'churn_probability', 'churn_risk']


In [9]:
df_active["TotalCharges"] = pd.to_numeric(
    df_active["TotalCharges"],
    errors="coerce"
)

df_active["TotalCharges"] = df_active["TotalCharges"].fillna(0)

print(df_active["TotalCharges"].isna().sum())

0


In [10]:
X_ltv = df_active[["tenure", "MonthlyCharges"]]
y_ltv = df_active["TotalCharges"]

print("X_ltv shape:", X_ltv.shape)
print("y_ltv shape:", y_ltv.shape)

X_ltv shape: (7043, 2)
y_ltv shape: (7043,)


In [11]:
from sklearn.model_selection import train_test_split

X_ltv_train, X_ltv_test, y_ltv_train, y_ltv_test = train_test_split(
    X_ltv,
    y_ltv,
    test_size=0.2,
    random_state=42
)

print("X_ltv_train:", X_ltv_train.shape)
print("X_ltv_test:", X_ltv_test.shape)
print("y_ltv_train:", y_ltv_train.shape)
print("y_ltv_test:", y_ltv_test.shape)

X_ltv_train: (5634, 2)
X_ltv_test: (1409, 2)
y_ltv_train: (5634,)
y_ltv_test: (1409,)


In [12]:
from sklearn.ensemble import RandomForestRegressor

ltv_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

ltv_model.fit(X_ltv_train, y_ltv_train)

print("LTV model trained successfully!")

LTV model trained successfully!


In [13]:
ltv_pred = ltv_model.predict(X_ltv_test)

print("First 10 predicted LTV values:")
print(ltv_pred[:10])

First 10 predicted LTV values:
[  24.77925    1013.33875    1055.1245       76.361      3419.8385
 6130.98870833 1761.4535     5067.484      7299.6835       19.4015    ]


In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_ltv_test, ltv_pred)
rmse = np.sqrt(mean_squared_error(y_ltv_test, ltv_pred))
r2 = r2_score(y_ltv_test, ltv_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE: 53.215759849713756
RMSE: 80.54655943219909
R2 Score: 0.998752934409777


In [15]:
comparison = pd.DataFrame({
    "Actual_LTV": y_ltv_test.values,
    "Predicted_LTV": ltv_pred
})

print(comparison.head(10))

   Actual_LTV  Predicted_LTV
0       24.80      24.779250
1      996.45    1013.338750
2     1031.70    1055.124500
3       76.35      76.361000
4     3260.10    3419.838500
5     6127.60    6130.988708
6     1759.40    1761.453500
7     5016.65    5067.484000
8     7250.15    7299.683500
9       19.40      19.401500


In [16]:
df_active["predicted_ltv"] = ltv_model.predict(
    df_active[["tenure", "MonthlyCharges"]]
)

print(
    df_active[
        ["customerID", "churn_probability", "churn_risk", "predicted_ltv"]
    ].head(10)
)

   customerID  churn_probability   churn_risk  predicted_ltv
0  7590-VHVEG           0.375392     Low Risk      29.904500
1  5575-GNVDE           0.042053     Low Risk    1892.904250
2  3668-QPYBK           0.504717  Medium Risk     106.529250
3  7795-CFOCW           0.030689     Low Risk    1888.436000
4  9237-HQITU           0.776983    High Risk     141.281988
5  9305-CDSKC           0.860548    High Risk     830.274000
6  1452-KIOVK           0.496953  Medium Risk    1928.617500
7  6713-OKOMC           0.259115     Low Risk     295.713250
8  7892-POOKP           0.739248    High Risk    2985.475750
9  6388-TABGU           0.011584     Low Risk    3485.589750


In [17]:
def customer_segment(row):
    if row["churn_probability"] >= 0.70 and row["predicted_ltv"] >= df_active["predicted_ltv"].median():
        return "High Value - High Risk"
    elif row["churn_probability"] >= 0.70:
        return "Low Value - High Risk"
    elif row["predicted_ltv"] >= df_active["predicted_ltv"].median():
        return "High Value - Low Risk"
    else:
        return "Low Value - Low Risk"


df_active["customer_segment"] = df_active.apply(
    customer_segment,
    axis=1
)

print(df_active["customer_segment"].value_counts())

customer_segment
High Value - Low Risk     3356
Low Value - Low Risk      2859
Low Value - High Risk      662
High Value - High Risk     166
Name: count, dtype: int64


In [18]:
priority_customers = df_active[
    df_active["customer_segment"] == "High Value - High Risk"
].sort_values(
    "predicted_ltv",
    ascending=False
)

print(
    priority_customers[
        [
            "customerID",
            "churn_probability",
            "churn_risk",
            "predicted_ltv",
            "customer_segment"
        ]
    ].head(20)
)

      customerID  churn_probability churn_risk  predicted_ltv  \
3330  8276-MQBYC           0.706261  High Risk    4924.334000   
6453  8634-MPHTR           0.724472  High Risk    4805.728500   
4960  7480-QNVZJ           0.707866  High Risk    4789.643817   
3950  5655-JSMZM           0.712012  High Risk    4755.379979   
3681  4433-JCGCG           0.727806  High Risk    4668.829750   
6914  7142-HVGBG           0.727862  High Risk    4425.596000   
3829  8374-XGEJJ           0.724425  High Risk    4423.911500   
3334  0337-CNPZE           0.703049  High Risk    4413.980500   
1478  9851-QXEEQ           0.709823  High Risk    4308.186750   
5131  5980-NOPLP           0.711708  High Risk    4272.792500   
3673  1723-HKXJQ           0.702847  High Risk    4255.662500   
1125  8111-SLLHI           0.742773  High Risk    4195.689000   
3775  1866-RZZQS           0.753293  High Risk    4119.868250   
808   4289-DTDKW           0.715675  High Risk    4083.367250   
2948  2845-AFFTX         

In [19]:
df_active.to_csv(
    "../data/customer_ltv_segments.csv",
    index=False
)

print("Final customer segmentation saved successfully!")

Final customer segmentation saved successfully!
